# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR^2 protein dataset using the `mlcroissant` library. We review record sets, their fields and associated `@id`s, extract and process tabular data, and perform exploratory data analysis.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and records from Croissant using `mlcroissant`. The `mlcroissant.Dataset` object gives access to all the metadata and makes it easy to load records from record sets.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Print dataset metadata
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")
print(f"Version: {meta.version}, Published: {meta.datePublished}, DOI: {meta.identifier}")
print(f"License: {meta.license}")
print(f"Keywords: {', '.join(meta.keywords) if meta.keywords else ''}")

## 2. Data Overview

Record sets define major tabular entities in this dataset. Each record set, field/column, and file object has an `@id`. Let's enumerate available record sets and show their field structure.

**Note**: All entities are referenced by `@id` for reproducible access.

In [ ]:
# List all record sets and their fields/columns by @id

record_sets = []
print("\nAvailable Record Sets and Fields:")
for rs in dataset.record_sets:
    print(f"- Record Set @id: {rs.id}")
    record_sets.append(rs.id)
    # List fields
    print("  Fields / Columns:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id} | name: {field.name} | data type: {field.data_type}")
    print("")
# For reference, print all record set ids
print("All RecordSet @ids:", record_sets)

## 3. Data Extraction

Now, extract records from a specific record set into a DataFrame. *Replace* the sample record set ID and field IDs with the actual ones displayed in the cell above.

For reproducibility, all IDs are managed via variables.

In [ ]:
# Set record set and select field IDs to load (replace with those from above if needed)
# Example: 'https://api.app.sen.science/frontiers/7154140/PROTEINS_1b7bd00d' as a likely table @id

record_set_id = record_sets[0] if record_sets else None  # Use the first record set, change if needed

# Load all records from the specified record set
if record_set_id is None:
    print("No record sets found in the schema!")
else:
    records = list(dataset.records(record_set=record_set_id))
    # Make DataFrame
    df = pd.DataFrame(records)
    print(f"Data loaded from RecordSet: {record_set_id}\nColumns: {df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)

Let's demonstrate common analytical tasks: filtering records, normalizing, and grouping by key fields. All fields referenced use their `@id` as column names.

- **Filtering:** For proteins with molecular weight (`MW`) or coverage above a threshold.
- **Normalization:** Standard scaling (z-score) of numeric fields.
- **Grouping:** Mean coverage or abundance by protein class or accession type.

Please replace `<numeric_field_id>` and `<group_field_id>` with actual `@id`s as enumerated in section 2 and 3.

In [ ]:
# Example field @id candidates (update as needed)
# For this dataset: likely IDs for numeric fields are for molecular weight, 'coverage', abundance, etc.

numeric_field = None
group_field = None
for c in df.columns:
    if 'weight' in c.lower() or 'coverage' in c.lower() or 'abundance' in c.lower():
        # Pick the first numeric-like field
        numeric_field = c
        break
for c in df.columns:
    if 'accession' in c.lower() or 'protein' in c.lower() or 'class' in c.lower():
        group_field = c
        break

print(f"Using numeric field: {numeric_field}")
print(f"Using group field: {group_field}\n")

# Ensure numeric
if numeric_field and df[numeric_field].dtype.kind in 'O':
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

threshold = 50000 if numeric_field else 0
if numeric_field:
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records where {numeric_field} > {threshold}: {len(filtered_df)} rows")
    display(filtered_df.head())

    # Normalize
    filtered_df[numeric_field + '_normalized'] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        filtered_df[numeric_field].std()
    )
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

    # Grouped analysis
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(by=numeric_field, ascending=False)
        print(f"\nMean {numeric_field} grouped by {group_field} (top 5):")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for analysis.")

## 5. Visualization

Visualize the selected numeric field's distribution and its variation by the chosen group. This aids in understanding both central tendency and spread across groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=40, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    
    if group_field:
        plt.figure(figsize=(10,6))
        # Show only top N groups by count
        value_counts = df[group_field].value_counts()
        top_groups = value_counts[:8].index
        filtered_vis = df[df[group_field].isin(top_groups)]
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_vis)
        plt.title(f"{numeric_field} across {group_field} (Top groups)")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion

Using `mlcroissant`, we:
- Loaded comprehensive protein mass spectrometry data using the Croissant schema,
- Explored record set and field structure with full `@id` traceability,
- Performed EDA: filtered, normalized, and grouped data on key quantitative fields,
- Visualized field distributions and group variations.

This FAIR^2 dataset is now ready for further biological discovery, machine learning, or domain analysis.